# GENETIC RELATIONSHIP MATRIX: STEP 04

This script calculates the GRM for the African, American, and European ancestry groups using Sparse GRM, 5,000 randomly selected markers, a minor allele frequency (MAF) > 0.01, and a relatedness cutoff of 0.05.

In [ ]:
# Load packages 
import os

In [ ]:
# Set environment variable for workspace bucket 
my_bucket = os.getenv('WORKSPACE_BUCKET')
# Workspace bucket 
my_bucket


# TEST COMMAND 

In [ ]:
%%writefile createSparseGRM_help.sh 

#!/bin/bash
set -o errexit
set -o nounset
# Print help message for createSparseGRM.R
createSparseGRM.R --help > "${output}"


In [ ]:
!gsutil cp createSparseGRM_help.sh {my_bucket}/data/scripts/

In [ ]:
!gsutil cat {my_bucket}/data/scripts/createSparseGRM_help.sh | head -n 1000

# TEST SUBMIT COMMAND 

In [ ]:
%%bash --out createSparseGRM_help  

#############################################################################
# DSUB: SUBMIT SCRIPT FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Source the dsub setup script, which likely contains environment setup and utility functions
source ~/aou_dsub.bash 

#############################################################################
# SUBMIT DSUB SCRIPT JOB FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Use a prebuilt container with PLINK2
# Set total disk space required 
# Define logging directory for dsub execution logs
# Input PLINK files (BED, BIM, FAM) from Google Cloud Storage (GCS)
# Provide cohort sample lists (recursively input from GCS)
# Define output directory for variant QC results (recursively stores outputs)
# Set environment variables for the script
# Specify the script to execute within the container

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.4.4 \
  --disk-size 2 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --output output="${WORKSPACE_BUCKET}/data/saige_step00/test/createSparseGRM_help.txt" \
  --script "${WORKSPACE_BUCKET}/data/scripts/createSparseGRM_help.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

# COMMAND 

In [ ]:
%%writefile EUR_create_sparse_grm.sh

#!/bin/bash
set -o errexit
set -o nounset
# Run createSparseGRM for EUR-like ancestry 
createSparseGRM.R \
--plinkFile="${input_path}/EUR_pruned_for_grm"  \
--nThreads=16  \
--outputPrefix="${output_path}/EUR_GRM" \
--numRandomMarkerforSparseKin=5000 \
--relatednessCutoff=0.05


In [ ]:
!gsutil cp EUR_create_sparse_grm.sh {my_bucket}/data/scripts/

In [ ]:
!gsutil cat {my_bucket}/data/scripts/EUR_create_sparse_grm.sh | head -n 1000

# SUMBIT COMMAND 

In [ ]:
%%bash --out eur_grm  

#############################################################################
# DSUB: SUBMIT SCRIPT FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Source the dsub setup script, which likely contains environment setup and utility functions
source ~/aou_dsub.bash 

#############################################################################
# SUBMIT DSUB SCRIPT JOB FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Use a prebuilt container with PLINK2
# Set total disk space required 
# Define logging directory for dsub execution logs
# Input PLINK files (BED, BIM, FAM) from Google Cloud Storage (GCS)
# Provide cohort sample lists (recursively input from GCS)
# Define output directory for variant QC results (recursively stores outputs)
# Set environment variables for the script
# Specify the script to execute within the container

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --disk-size 200 \
  --min-ram 14 \
  --min-cores 16 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive input_path="${WORKSPACE_BUCKET}/data/saige_step00/grm_plink_files" \
  --output-recursive output_path="${WORKSPACE_BUCKET}/data/saige_step00/grm" \
  --script "${WORKSPACE_BUCKET}/data/scripts/EUR_create_sparse_grm.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################